# Prepare K-Hairstyle Attribute Dataset

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from app.ml.datasets import (
    build_attribute_records_from_khairstyle,
    build_label_vocab,
    train_val_split,
    write_jsonl_manifest,
)
from app.ml.khairstyle_translation import (
    FIELD_TRANSLATIONS,
    build_normalized_attributes,
    repair_mojibake,
    translate_labels,
)

RAW_LABEL_ROOT = BACKEND_ROOT / 'data' / 'raw' / 'khairstyle' / 'mqset' / 'labels' / 'labels_mqset'
PROCESSED_METADATA_ROOT = BACKEND_ROOT / 'data' / 'processed' / 'hairstyle_assets' / 'metadata'
DATASET_DIR = BACKEND_ROOT / 'data' / 'datasets' / 'hairstyle_attribute'

print('Project root:', PROJECT_ROOT)
print('Raw label root exists:', RAW_LABEL_ROOT.exists())
print('Processed metadata root exists:', PROCESSED_METADATA_ROOT.exists())


Project root: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon
Raw label root exists: True
Processed metadata root exists: True


In [2]:
pd.Series({field: len(values) for field, values in FIELD_TRANSLATIONS.items()}).sort_values(ascending=False)


basestyle         20
curl               9
partition          8
color              7
bang               7
length             5
natural-curl       4
damage             4
hair-width         3
loss               3
side               3
basestyle-type     2
sex                2
before-after       2
vertical           2
dtype: int64

In [3]:
raw_json_paths = sorted(RAW_LABEL_ROOT.rglob('*.json'))
sample_raw_path = raw_json_paths[0]
sample_raw_payload = json.loads(sample_raw_path.read_text(encoding='utf-8', errors='replace'))

sample_fields = ['basestyle', 'basestyle-type', 'length', 'curl', 'bang', 'side', 'color', 'partition', 'sex']
pd.DataFrame(
    {
        'raw_value': [sample_raw_payload.get(field) for field in sample_fields],
        'repaired_value': [repair_mojibake(sample_raw_payload.get(field)) for field in sample_fields],
    },
    index=sample_fields,
)


,raw_value,repaired_value
basestyle,가르마,가르마
basestyle-type,단,단
length,남자,남자
curl,X,X
bang,기타(남자 내림머리),기타(남자 내림머리)
side,원블럭,원블럭
color,블랙,블랙
partition,5:5,5:5
sex,남,남


In [4]:
sample_translated = translate_labels(sample_raw_payload)
sample_normalized = build_normalized_attributes(sample_translated)

display(pd.Series(sample_translated, name='translated_labels'))
display(pd.Series(sample_normalized, name='normalized_attributes'))


basestyle                        parted
basestyle-type               short_type
length                        men_style
curl                    straight_code_x
bang              other_mens_down_style
loss                       no_hair_loss
side                          one_block
color                             black
partition                      part_5_5
sex                                male
before-after                      after
vertical                          upper
hair-width                       medium
natural-curl                 semi_curly
damage                           virgin
Name: translated_labels, dtype: object

length              short
curl             straight
bang                 side
volume             medium
side_hair         covered
color               black
style_family    side_part
Name: normalized_attributes, dtype: object

In [5]:
coverage_rows = []
for path in raw_json_paths[:500]:
    payload = json.loads(path.read_text(encoding='utf-8', errors='replace'))
    translated = translate_labels(payload)
    coverage_rows.append(
        {
            'path': str(path.relative_to(PROJECT_ROOT)).replace('\\', '/'),
            'unmapped_fields': [field for field, value in translated.items() if value.startswith('unmapped_')],
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
coverage_df['unmapped_count'] = coverage_df['unmapped_fields'].apply(len)
coverage_df['unmapped_count'].value_counts().sort_index()


unmapped_count
0    500
Name: count, dtype: int64

In [6]:
coverage_df[coverage_df['unmapped_count'] > 0].head(20)


,path,unmapped_fields,unmapped_count


In [7]:
MAX_RECORDS = None
TRAIN_RATIO = 0.85
RANDOM_SEED = 42
TRAIN_MANIFEST = DATASET_DIR / 'train.jsonl'
VAL_MANIFEST = DATASET_DIR / 'val.jsonl'
VOCAB_PATH = DATASET_DIR / 'label_vocab.json'
SUMMARY_PATH = DATASET_DIR / 'summary.json'


In [8]:
records = build_attribute_records_from_khairstyle(limit=MAX_RECORDS)
train_records, val_records = train_val_split(records, train_ratio=TRAIN_RATIO, seed=RANDOM_SEED)
label_vocab = build_label_vocab(records)

write_jsonl_manifest(train_records, TRAIN_MANIFEST)
write_jsonl_manifest(val_records, VAL_MANIFEST)
VOCAB_PATH.parent.mkdir(parents=True, exist_ok=True)
VOCAB_PATH.write_text(json.dumps(label_vocab, indent=2, ensure_ascii=False), encoding='utf-8')

summary = {
    'total_records': len(records),
    'train_records': len(train_records),
    'val_records': len(val_records),
    'label_fields': list(label_vocab.keys()),
    'label_class_counts': {field: len(values) for field, values in label_vocab.items()},
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
pd.Series(summary)


total_records                                                        76
train_records                                                        64
val_records                                                          12
label_fields          [length, curl, bang, volume, side_hair, color,...
label_class_counts    {'length': 3, 'curl': 4, 'bang': 5, 'volume': ...
dtype: object

In [9]:
pd.DataFrame(records[:5])


,image_path,labels,source_id,source_dataset
0,backend/data/processed/hairstyle_assets/images...,"{'length': 'short', 'curl': 'straight', 'bang'...",hair_000001,K-Hairstyle
1,backend/data/processed/hairstyle_assets/images...,"{'length': 'short', 'curl': 'wavy', 'bang': 's...",hair_000002,K-Hairstyle
2,backend/data/processed/hairstyle_assets/images...,"{'length': 'short', 'curl': 'straight', 'bang'...",hair_000003,K-Hairstyle
3,backend/data/processed/hairstyle_assets/images...,"{'length': 'short', 'curl': 'wavy', 'bang': 's...",hair_000004,K-Hairstyle
4,backend/data/processed/hairstyle_assets/images...,"{'length': 'short', 'curl': 'straight', 'bang'...",hair_000005,K-Hairstyle
